# PIMCO AI Surveillance Analysis Based on ABS Prospectuses

### Milestones
- **Milestone 1:** Data collection, parsing, metadata filtering, and keyword/regex search
- **Milestone 2:** Embedding-based semantic search and Q&A
- **Milestone 3:** User interface and evaluation

In [108]:
import os
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup as bsoup

In [109]:
from pathlib import Path

# Should get the project root
PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)

Project root: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses


In [110]:
raw_data = PROJECT_ROOT / "data" / "raw"
processed_data = PROJECT_ROOT / "data" / "processed"
manifest_path = PROJECT_ROOT / "data" / "manifest.csv"

# Output file
output = processed_data / "abs_index.csv"

print("Raw data:", raw_data)
print("Manifest:", manifest_path)
print("Processed data:", processed_data)
print("Output file:", output)

Raw data: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/raw
Manifest: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/manifest.csv
Processed data: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed
Output file: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_index.csv


## Milestone 1 — Data Collection and Text Extraction

The dataset contains SEC ABS prospectus filings in HTML (`.htm`) format.

For each filing, we:
1. Locate the HTML file
2. Read the HTML
3. Remove HTML formatting
4. Extract readable text
5. Store the result in a DataFrame

In [111]:
def get_htm_files(data_folder):
    data_folder = Path(data_folder)
    
    if not data_folder.exists():
        raise FileNotFoundError(
            f"Data folder not found: {data_folder}"
        )
    
    htm_files = sorted(
        file
        for file in data_folder.iterdir()
        if file.is_file() and file.suffix.lower() == ".htm"
    )
    
    return htm_files


htm_files = get_htm_files(raw_data)

print(f"Found {len(htm_files)} HTM filings.")

Found 122 HTM filings.


In [112]:
# Extract readable text from an SEC HTML filing.    
def extract_text_from_htm(file_path):
   with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:
        
        html = file.read()
        soup = bsoup(
        html,
        "html.parser"
    )
        text = soup.get_text(
        " ",
        strip=True
    )
        return text

In [113]:
data = []

for file_path in htm_files:
    
    text = extract_text_from_htm(
        file_path
    )
    
    data.append({
        "file_name": file_path.name,
        "extracted_text": text
    })

df = pd.DataFrame(data)

print("Number of filings:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Number of filings: 122
Number of columns: 2


,file_name,extracted_text
0,AFS_SENSUB_CORP__2024-05-15_424H.htm,424H 1 d679938d424h.htm 424H 424H Table of Con...
1,AFS_SENSUB_CORP__2024-06-26_424H.htm,424H 1 d792078d424h.htm 424H 424H Table of Con...
2,AFS_SENSUB_CORP__2024-10-02_424H.htm,424H 1 d847362d424h.htm 424H 424H Table of Con...
3,AFS_SENSUB_CORP__2025-01-06_424H.htm,424H 1 d909372d424h.htm 424H 424H Table of Con...
4,AFS_SENSUB_CORP__2025-05-01_424H.htm,424H 1 d111258d424h.htm 424H 424H Table of Con...


In [114]:
# metadata
def filename_metadata(df):
    df = df.copy()
    # Filing date
    df["filing_date"] = df["file_name"].str.extract(
        r"(\d{4}-\d{2}-\d{2})"
    )
    # Filing type
    df["filing_type"] = df["file_name"].str.extract(
        r"_(424H)\.htm$"
    )
    # Deal name
    df["deal_name"] = (
        df["file_name"]
        .str.replace(
            r"_\d{4}-\d{2}-\d{2}_424H\.htm$",
            "",
            regex=True
        )
        .str.replace(
            "_",
            " ",
            regex=False
        )
    )
    return df
df = filename_metadata(df)
df[
    [
        "file_name",
        "deal_name",
        "filing_date",
        "filing_type"
    ]
].tail()

,file_name,deal_name,filing_date,filing_type
117,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2025-10-01_424...,WORLD OMNI AUTO RECEIVABLES LLC,2025-10-01,424H
118,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-02-02_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-02-02,424H
119,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-04-29_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-04-29,424H
120,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-07-13_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-07-13,424H
121,WORLD_OMNI_AUTO_RECEIVABLES_LLC_2026-08-12_424...,WORLD OMNI AUTO RECEIVABLES LLC,2026-08-12,424H


In [115]:
# Read manifest csv
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Manifest not found: {manifest_path}"
    )
manifest = pd.read_csv(
    manifest_path
)
print("Manifest rows:", len(manifest))
print("Manifest columns:")
print(manifest.columns.tolist())

# Get actual filename from local_path
manifest["file_name"] = (
    manifest["local_path"]
    .astype(str)
    .str.split(r"[\\/]")
    .str[-1]
)

Manifest rows: 129
Manifest columns:
['entity_name', 'cik', 'form', 'filing_date', 'accession', 'accession_clean', 'primary_doc', 'description', 'doc_url', 'index_url', 'local_path', 'downloaded']


In [116]:
manifest["file_name"] = (
    manifest["local_path"]
    .astype(str)
    .str.split(r"[\\/]")
    .str[-1]
)

In [117]:
# Remove old SEC metadata columns from df
metadata_columns = [
    "cik",
    "accession",
    "primary_doc",
    "doc_url",
    "cik_x",
    "cik_y",
    "accession_x",
    "accession_y",
    "primary_doc_x",
    "primary_doc_y",
    "doc_url_x",
    "doc_url_y"
]
df = df.drop(
    columns=[
        col for col in metadata_columns
        if col in df.columns
    ],
    errors="ignore"
)

# Keep one manifest row per filename
manifest_unique = (
    manifest
    .drop_duplicates("file_name")
)
print(
    "Unique manifest filings:",
    len(manifest_unique)
)

# Merge SEC metadata into df
df = df.merge(
    manifest_unique[
        [
            "file_name",
            "cik",
            "accession",
            "primary_doc",
            "doc_url"
        ]
    ],
    on="file_name",
    how="left"
    )
print(
        "SEC metadata matched:",
        df["cik"].notna().sum(),
        "/",
        len(df)
    )

Unique manifest filings: 122
SEC metadata matched: 122 / 122


In [118]:
# Normalize whitespace in extracted filing text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    return text.strip()

In [119]:
df["clean_text"] = (
    df["extracted_text"]
    .apply(clean_text)
)

In [120]:
# Character count
df["text_length"] = (
    df["clean_text"]
    .str.len()
)
# Word count
df["word_count"] = (
    df["clean_text"]
    .str.split()
    .str.len()
)
# Convert filing date to datetime
df["filing_date"] = pd.to_datetime(
    df["filing_date"],
    errors="coerce"
)
# Extract filing year
df["filing_year"] = (
    df["filing_date"]
    .dt.year
)

In [121]:

print("DATA QUALITY CHECK: ")
print(
    "Total filings:",
    len(df)
)
print(
    "Empty extracted text:",
    (df["text_length"] == 0).sum()
)
print(
    "Missing filing dates:",
    df["filing_date"].isna().sum()
)
print(
    "Missing deal names:",
    df["deal_name"].isna().sum()
)
print(
    "Missing CIK:",
    df["cik"].isna().sum()
)
print(
    "Missing SEC URLs:",
    df["doc_url"].isna().sum()
)

print(
    "Average words per filing:",
    round(df["word_count"].mean())
)
print(
    "Shortest filing:",
    df["word_count"].min(),
    "words"
)
print(
    "Longest filing:",
    df["word_count"].max(),
    "words"
)

DATA QUALITY CHECK: 
Total filings: 122
Empty extracted text: 0
Missing filing dates: 0
Missing deal names: 0
Missing CIK: 0
Missing SEC URLs: 0
Average words per filing: 147154
Shortest filing: 83055 words
Longest filing: 586179 words


In [122]:
df.to_csv(
    output,
    index=False
)
print(
    f"Saved processed index to:\n{output}"
)
print(
    f"Number of filings: {len(df)}"
)
print(
    f"Number of columns: {len(df.columns)}"
)

Saved processed index to:
/Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_index.csv
Number of filings: 122
Number of columns: 13


In [123]:
def extract_snippets(
    text,
    pattern,
    snippets_per_result=2,
    context_chars=250
):
    """
    Extract text surrounding each search match.
    """
    
    if not isinstance(text, str):
        return []
    
    try:
        compiled_pattern = re.compile(
            pattern,
            flags=re.IGNORECASE
        )
    except re.error:
        return []
    
    snippets = []
    
    for match in compiled_pattern.finditer(text):
        
        start = max(
            0,
            match.start() - context_chars
        )
        
        end = min(
            len(text),
            match.end() + context_chars
        )
        
        snippet = text[start:end].strip()
        
        if start > 0:
            snippet = "..." + snippet
        
        if end < len(text):
            snippet += "..."
        
        snippets.append(snippet)
        
        if len(snippets) >= snippets_per_result:
            break
    
    return snippets

In [124]:
def search_filings(
    query,
    year=None,
    filing_type=None,
    regex=False,
    max_results=10,
    snippets_per_result=2
):
    """
    Search ABS prospectuses using keywords,
    phrases, or regular expressions.
    """
    
    # Start with the processed dataset
    search_df = df.copy()
    
    # Metadata filtering
    if year is not None:
        
        search_df = search_df[
            search_df["filing_year"] == int(year)
        ]
    
    if filing_type is not None:
        
        search_df = search_df[
            search_df["filing_type"]
            .fillna("")
            .str.lower()
            ==
            str(filing_type).lower()
        ]
    
    # Create search pattern
    if regex:
        pattern = query
    
    else:
        pattern = re.escape(query)
    
    try:
        compiled_pattern = re.compile(
            pattern,
            flags=re.IGNORECASE
        )
    
    except re.error as error:
        raise ValueError(
            f"Invalid regular expression: {error}"
        )

    # Count matches
    search_df["matches"] = (
        search_df["clean_text"]
        .fillna("")
        .apply(
            lambda text:
            len(
                compiled_pattern.findall(text)
            )
        )
    )
    
    # Keep only matching filings
    search_df = search_df[
        search_df["matches"] > 0
    ].copy()
    
    # Supporting snippets
    search_df["supporting_snippets"] = (
        search_df["clean_text"]
        .fillna("")
        .apply(
            lambda text:
            extract_snippets(
                text,
                pattern,
                snippets_per_result
            )
        )
    )
    
    # Rank results
    search_df = search_df.sort_values(
        by="matches",
        ascending=False
    )
    
    # Return useful columns
    columns = [
        "deal_name",
        "filing_date",
        "filing_year",
        "filing_type",
        "matches",
        "supporting_snippets",
        "doc_url",
        "file_name"
    ]
    
    return (
        search_df[columns]
        .head(max_results)
        .reset_index(drop=True)
    )

In [125]:
results = search_filings(
    "custody",
    max_results=10
)
results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-03-23,2026,424H,15,[...assu Whole Loans 240 Certain Rights of eac...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-15,2026,424H,13,[...ssu Whole Loans 246 Certain Rights of each...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
6,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-05,2026,424H,10,[...Passu Whole Loans 243 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
7,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
8,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-02-11,2026,424H,9,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
9,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-06-10,2026,424H,8,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...


In [126]:
results = search_filings(
    "physical custody",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name


In [127]:
results = search_filings(
    r"\b(physical|paper)\b.{0,100}\b(custody|custodian)\b",
    regex=True,
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,SANTANDER DRIVE AUTO RECEIVABLES LLC,2026-02-11,2026,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2026-02-1...
1,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-11-06,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-11-0...
2,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-09-10,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-09-1...
3,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-07-17,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-07-1...
4,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-05-14,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-05-1...
5,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-03-12,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-03-1...
6,SANTANDER DRIVE AUTO RECEIVABLES LLC,2025-01-10,2025,424H,2,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2025-01-1...
7,Bridgecrest Auto Funding LLC,2026-07-15,2026,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/197482...,Bridgecrest_Auto_Funding_LLC_2026-07-15_424H.htm
8,SANTANDER DRIVE AUTO RECEIVABLES LLC,2024-10-10,2024,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2024-10-1...
9,SANTANDER DRIVE AUTO RECEIVABLES LLC,2024-09-11,2024,424H,1,[...any other electronic records relating to t...,https://www.sec.gov/Archives/edgar/data/138309...,SANTANDER_DRIVE_AUTO_RECEIVABLES_LLC_2024-09-1...


In [128]:
results = search_filings(
    "custody",
    year=2025,
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,BMW AUTO LEASING LLC,2025-05-29,2025,424H,6,[...s 128 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-05-29_424H.htm
6,BMW AUTO LEASING LLC,2025-10-02,2025,424H,6,[...s 129 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-10-02_424H.htm
7,HARLEY-DAVIDSON CUSTOMER FUNDING CORP,2025-05-14,2025,424H,3,[...e Uniform Commercial Code. As described un...,https://www.sec.gov/Archives/edgar/data/111492...,HARLEY-DAVIDSON_CUSTOMER_FUNDING_CORP__2025-05...
8,CARMAX AUTO FUNDING LLC,2025-09-09,2025,424H,2,[...r of this securitization transaction and w...,https://www.sec.gov/Archives/edgar/data/125938...,CARMAX_AUTO_FUNDING_LLC_2025-09-09_424H.htm
9,TOYOTA AUTO FINANCE RECEIVABLES LLC,2025-10-08,2025,424H,2,[...behalf of their participants through their...,https://www.sec.gov/Archives/edgar/data/113113...,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2025-10-08...


In [129]:
results = search_filings(
    "custody",
    filing_type="424H",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-03-23,2026,424H,15,[...assu Whole Loans 240 Certain Rights of eac...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-15,2026,424H,13,[...ssu Whole Loans 246 Certain Rights of each...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
6,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-05-05,2026,424H,10,[...Passu Whole Loans 243 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
7,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
8,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-02-11,2026,424H,9,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
9,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2026-06-10,2026,424H,8,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...


In [130]:
results = search_filings(
    "custody",
    year=2025,
    filing_type="424H",
    max_results=10
)

results

,deal_name,filing_date,filing_year,filing_type,matches,supporting_snippets,doc_url,file_name
0,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-01,2025,424H,16,[...Passu Whole Loans 255 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
1,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-17,2025,424H,15,[...Passu Whole Loans 238 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
2,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-12-08,2025,424H,14,[...Passu Whole Loans 276 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
3,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-09-30,2025,424H,11,"[...r under such Non-Serviced PSA, entitlement...",https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
4,WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC,2025-11-13,2025,424H,10,[...Shift Whole Loans 249 Certain Rights of ea...,https://www.sec.gov/Archives/edgar/data/850779...,WELLS_FARGO_COMMERCIAL_MORTGAGE_SECURITIES_INC...
5,BMW AUTO LEASING LLC,2025-05-29,2025,424H,6,[...s 128 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-05-29_424H.htm
6,BMW AUTO LEASING LLC,2025-10-02,2025,424H,6,[...s 129 Sales Proceeds and Termination Proce...,https://www.sec.gov/Archives/edgar/data/112653...,BMW_AUTO_LEASING_LLC_2025-10-02_424H.htm
7,HARLEY-DAVIDSON CUSTOMER FUNDING CORP,2025-05-14,2025,424H,3,[...e Uniform Commercial Code. As described un...,https://www.sec.gov/Archives/edgar/data/111492...,HARLEY-DAVIDSON_CUSTOMER_FUNDING_CORP__2025-05...
8,CARMAX AUTO FUNDING LLC,2025-09-09,2025,424H,2,[...r of this securitization transaction and w...,https://www.sec.gov/Archives/edgar/data/125938...,CARMAX_AUTO_FUNDING_LLC_2025-09-09_424H.htm
9,TOYOTA AUTO FINANCE RECEIVABLES LLC,2025-10-08,2025,424H,2,[...behalf of their participants through their...,https://www.sec.gov/Archives/edgar/data/113113...,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2025-10-08...


In [131]:
test_queries = [
    "physical",
    "paper",
    "custody",
    "custodian"
]

for query in test_queries:
    
    results = search_filings(
        query,
        max_results=5
    )
    
    print(f"QUERY: {query}")
    print(f"MATCHING FILINGS: {len(results)}")
    
    if len(results) > 0:
        
        top_result = results.iloc[0]
        
        print(
            "TOP DEAL:",
            top_result["deal_name"]
        )
        
        print(
            "DATE:",
            top_result["filing_date"]
        )
        
        print(
            "MATCHES:",
            top_result["matches"]
        )
        
        print("\nSNIPPET:")
        
        print(
            top_result["supporting_snippets"][0]
        )
        
    else:
        
        print("No results found.")

QUERY: physical
MATCHING FILINGS: 5
TOP DEAL: WELLS FARGO COMMERCIAL MORTGAGE SECURITIES INC
DATE: 2026-02-11 00:00:00
MATCHES: 59

SNIPPET:
...ptible to damage or wear and tear than other types of multifamily housing, the reliance on the financial well-being of the college or university to which it relates, competition from on campus housing units, which may adversely affect occupancy, the physical layout of the housing, which may not be readily convertible to traditional multifamily use, and that student tenants have a higher turnover rate than other types of multifamily tenants, which in certain cases is compounded by the fact that student l...
QUERY: paper
MATCHING FILINGS: 5
TOP DEAL: BMW AUTO LEASING LLC
DATE: 2025-05-29 00:00:00
MATCHES: 19

SNIPPET:
...eness of the settlement pending PIMCO’s appeal of its objection. The case has been referred to the circuit mediator, and, on February 24, 2025, the Consumer Financial Protection Bureau sought extension of time for 90 days to prep

In [132]:
df.head()
df.columns.tolist()

['file_name',
 'extracted_text',
 'filing_date',
 'filing_type',
 'deal_name',
 'cik',
 'accession',
 'primary_doc',
 'doc_url',
 'clean_text',
 'text_length',
 'word_count',
 'filing_year']

## Milestone 2

In [133]:
import pandas as pd
import re
import json
import gzip
import hashlib
from pathlib import Path

# Reuse the processed_data path from Milestone 1
index_path = processed_data / "abs_index.csv"

df = pd.read_csv(index_path)

print("Total filings:", len(df))
print("Columns:", df.columns.tolist())

Total filings: 122
Columns: ['file_name', 'extracted_text', 'filing_date', 'filing_type', 'deal_name', 'cik', 'accession', 'primary_doc', 'doc_url', 'clean_text', 'text_length', 'word_count', 'filing_year']


In [134]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75

def chunk_document(row, chunk_size=500, overlap=75):
    text = str(row["clean_text"])

    if not text.strip() or text == "nan":
        return []

    # Find each word and its position in the original text
    words = list(re.finditer(r"\S+", text))
    total_words = len(words)

    step = chunk_size - overlap

    # Create a unique ID for each filing
    filename = str(row["file_name"])
    accession = str(row["accession"])

    file_hash = hashlib.sha256(
        filename.encode("utf-8")
    ).hexdigest()[:10]

    filing_id = f"{accession}_{file_hash}"

    chunks = []
    previous_end = 0

    for start in range(0, total_words, step):
        end = min(start + chunk_size, total_words)

        # Avoid creating a very small final chunk
        if (
            end == total_words
            and total_words > chunk_size
            and end - start < chunk_size // 2
        ):
            start = total_words - chunk_size

        if end <= previous_end:
            break

        # Get character positions
        char_start = words[start].start()
        char_end = words[end - 1].end()

        chunk_text = text[char_start:char_end]

        chunk = {
            "chunk_id": f"{filing_id}_chunk_{len(chunks)}",
            "filing_id": filing_id,
            "chunk_index": len(chunks),
            "file_name": filename,
            "deal_name": row["deal_name"],
            "filing_date": str(row["filing_date"]),
            "filing_year": int(row["filing_year"]),
            "filing_type": row["filing_type"],
            "cik": str(row["cik"]),
            "accession": accession,
            "doc_url": row["doc_url"],
            "word_start": start,
            "word_end": end,
            "char_start": char_start,
            "char_end": char_end,
            "word_count": end - start,
            "text": chunk_text
        }

        chunks.append(chunk)
        previous_end = end

        if end == total_words:
            break

    return chunks

In [135]:
output_path = processed_data / "abs_chunks.jsonl.gz"

total_chunks = 0
filings_processed = 0
empty_filings = 0

with gzip.open(
    output_path, "wt", encoding="utf-8"
) as f:

    for _, row in df.iterrows():

        chunks = chunk_document(
            row,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        if not chunks:
            empty_filings += 1
            continue

        # Check that the entire document was covered
        assert chunks[0]["word_start"] == 0
        assert chunks[-1]["word_end"] == len(
            str(row["clean_text"]).split()
        )

        for chunk in chunks:
            f.write(
                json.dumps(chunk, ensure_ascii=False, default=str)
                + "\n"
            )

        total_chunks += len(chunks)
        filings_processed += 1

print("Filings processed:", filings_processed)
print("Total chunks generated:", total_chunks)
print("Empty filings:", empty_filings)
print("Saved chunks to:", output_path)

Filings processed: 122
Total chunks generated: 42283
Empty filings: 0
Saved chunks to: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_chunks.jsonl.gz


In [136]:
with gzip.open(
    output_path, "rt", encoding="utf-8"
) as f:

    for i in range(3):
        chunk = json.loads(next(f))

        print("Chunk ID:", chunk["chunk_id"])
        print("Deal:", chunk["deal_name"])
        print("Filing date:", chunk["filing_date"])
        print("Word count:", chunk["word_count"])
        print("Source:", chunk["doc_url"])
        print()
        print("Text preview:")
        print(chunk["text"][:500])
        print()

Chunk ID: 0001193125-24-139394_6460085a42_chunk_0
Deal: AFS SENSUB CORP 
Filing date: 2024-05-15
Word count: 500
Source: https://www.sec.gov/Archives/edgar/data/1347185/000119312524139394/d679938d424h.htm

Text preview:
424H 1 d679938d424h.htm 424H 424H Table of Contents Filed pursuant to Rule 424(h) Registration Statement No. 333-261851 Information contained herein is subject to completion or amendment. A registration statement relating to these securities has been filed and declared effective with the Securities and Exchange Commission. These securities may not be sold without delivery of a final prospectus. This prospectus shall not constitute an offer to sell or the solicitation of an offer to buy nor shall

Chunk ID: 0001193125-24-139394_6460085a42_chunk_1
Deal: AFS SENSUB CORP 
Filing date: 2024-05-15
Word count: 500
Source: https://www.sec.gov/Archives/edgar/data/1347185/000119312524139394/d679938d424h.htm

Text preview:
- ● excess cashflow collected on the pool of automobile lo

In [137]:
chunk_ids = set()
total = 0
empty_chunks = 0
missing_urls = 0
duplicate_ids = 0

with gzip.open(
    output_path, "rt", encoding="utf-8"
) as f:

    for line in f:
        chunk = json.loads(line)
        total += 1

        if not chunk["text"].strip():
            empty_chunks += 1

        if not chunk["doc_url"]:
            missing_urls += 1

        if chunk["chunk_id"] in chunk_ids:
            duplicate_ids += 1

        chunk_ids.add(chunk["chunk_id"])

print("Total chunks:", total)
print("Empty chunks:", empty_chunks)
print("Missing source URLs:", missing_urls)
print("Duplicate chunk IDs:", duplicate_ids)

assert empty_chunks == 0
assert missing_urls == 0
assert duplicate_ids == 0


Total chunks: 42283
Empty chunks: 0
Missing source URLs: 0
Duplicate chunk IDs: 0


In [138]:
%pip install openai python-dotenv faiss-cpu tiktoken tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [139]:
import os
import json
import gzip
import hashlib

from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import tiktoken

from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# Existing Milestone 1 directory
processed_data = Path(processed_data).resolve()

# Repository root
repo_root = processed_data.parents[1]

# Load PIMCO API key from .env
load_dotenv(repo_root / ".env")

api_key = os.getenv("API_KEY")
assert api_key, "OPENAI_API_KEY not found in .env"

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("OPENAI_BASE_URL") or None,
    max_retries=2,
    timeout=90.0
)

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

print("API credentials loaded successfully.")
print("Embedding model:", EMBEDDING_MODEL)
print("Processed data directory:", processed_data)


API credentials loaded successfully.
Embedding model: text-embedding-3-small
Processed data directory: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed


In [140]:
chunk_path = processed_data / "abs_chunks.jsonl.gz"

chunks = []

with gzip.open(chunk_path, "rt", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

# Toyota April 2026 filing
TOYOTA_FILE = (
    "TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_"
    "2026-04-08_424H.htm"
)

toyota_chunks = [
    c for c in chunks
    if c["file_name"] == TOYOTA_FILE
]

other_chunks = [
    c for c in chunks
    if c["file_name"] != TOYOTA_FILE
]

# IMPORTANT: Keep this ordering unchanged.
# It matches the previously saved FAISS index.
ordered_chunks = toyota_chunks + other_chunks

print("Total chunks:", len(ordered_chunks))
print("Toyota chunks:", len(toyota_chunks))
print("Other chunks:", len(other_chunks))
print("First filing:", ordered_chunks[0]["file_name"])

Total chunks: 42283
Toyota chunks: 312
Other chunks: 41971
First filing: TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08_424H.htm


In [146]:
index_path = processed_data / "abs_openai_semantic.faiss"
state_path = processed_data / "abs_openai_semantic_state.json"

# Hash the dataset and chunk ordering
source_hash = hashlib.sha256(
    chunk_path.read_bytes()
).hexdigest()

order_hash = hashlib.sha256(
    "\n".join(
        c["chunk_id"] for c in ordered_chunks
    ).encode("utf-8")
).hexdigest()

if index_path.exists() and state_path.exists():

    state = json.loads(state_path.read_text())

    # Verify saved index configuration
    assert state["model"] == EMBEDDING_MODEL
    assert state["source_hash"] == source_hash
    assert state["order_hash"] == order_hash

    index = faiss.read_index(str(index_path))

    assert index.ntotal == state["indexed_chunks"]
    assert index.d == EMBEDDING_DIM

    print("Existing FAISS index loaded successfully.")

elif not index_path.exists() and not state_path.exists():

    index = faiss.IndexFlatIP(EMBEDDING_DIM)

    print("Created new FAISS index.")

else:
    raise RuntimeError(
        "Incomplete checkpoint. Check saved index files."
    )

print("\nIndexed chunks:", index.ntotal)
print("Total available chunks:", len(ordered_chunks))
print("Remaining:", len(ordered_chunks) - index.ntotal)
print("Embedding dimensions:", index.d)

toyota_indexed = min(index.ntotal, len(toyota_chunks))

print("\nToyota chunks indexed:",
      toyota_indexed, "/", len(toyota_chunks))


Existing FAISS index loaded successfully.

Indexed chunks: 2808
Total available chunks: 42283
Remaining: 39475
Embedding dimensions: 1536

Toyota chunks indexed: 312 / 312


In [147]:
# Keep False unless you want to generate more embeddings
RUN_MORE_EMBEDDINGS = False

# None means all remaining chunks
TARGET_CHUNKS = None

BATCH_SIZE = 24
SAVE_EVERY = 240
MAX_RUN_TOKENS = 1_000_000

encoding = tiktoken.encoding_for_model(EMBEDDING_MODEL)

def save_checkpoint():
    faiss.write_index(index, str(index_path))

    state = {
        "model": EMBEDDING_MODEL,
        "source_hash": source_hash,
        "order_hash": order_hash,
        "indexed_chunks": index.ntotal
    }

    state_path.write_text(
        json.dumps(state, indent=2)
    )

if RUN_MORE_EMBEDDINGS:
    target = (
        len(ordered_chunks)
        if TARGET_CHUNKS is None
        else min(TARGET_CHUNKS, len(ordered_chunks))
    )

    start_count = index.ntotal
    last_saved = index.ntotal

    estimated_run_tokens = 0
    actual_run_tokens = 0

    for start in tqdm(
        range(index.ntotal, target, BATCH_SIZE),
        desc="Creating OpenAI embeddings"
    ):

        end = min(start + BATCH_SIZE, target)

        batch_texts = [
            ordered_chunks[i]["text"]
            for i in range(start, end)
        ]

        batch_token_counts = [
            len(encoding.encode(
                text,
                disallowed_special=()
            ))
            for text in batch_texts
        ]

        assert max(batch_token_counts) <= 8192, (
            "Chunk exceeds embedding token limit."
        )

        batch_tokens = sum(batch_token_counts)

        if estimated_run_tokens + batch_tokens > MAX_RUN_TOKENS:
            print("Paused: per-run token budget reached.")
            break

        # API request
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch_texts
        )

        response_data = sorted(
            response.data,
            key=lambda item: item.index
        )

        vectors = np.array(
            [item.embedding for item in response_data],
            dtype="float32"
        )

        assert vectors.shape == (
            len(batch_texts), EMBEDDING_DIM
        )

        # Normalize for cosine similarity
        faiss.normalize_L2(vectors)

        # Add new embeddings to existing index
        index.add(vectors)

        estimated_run_tokens += batch_tokens

        if response.usage is not None:
            actual_run_tokens += response.usage.total_tokens

        # Save periodically
        if index.ntotal - last_saved >= SAVE_EVERY:
            save_checkpoint()
            last_saved = index.ntotal

    if index.ntotal != last_saved:
        save_checkpoint()

    print("\nEmbedding run completed.")
    print("New chunks embedded:",
          index.ntotal - start_count)
    print("Total indexed chunks:", index.ntotal)
    print("Estimated run tokens:", estimated_run_tokens)
    print("Reported API tokens:", actual_run_tokens)

else:
    print("Embedding generation skipped.")
    print("Existing indexed chunks:", index.ntotal)


Embedding generation skipped.
Existing indexed chunks: 2808


In [148]:
def semantic_search(
    question,
    top_k=8,
    file_name=None,
    year=None,
    unique_filings=False
):

    if index.ntotal == 0:
        raise ValueError("FAISS index is empty.")

    if top_k < 1:
        raise ValueError("top_k must be positive.")

    # Generate query embedding
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[question]
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    # Helper to format retrieved passages
    def format_results(hits):

        results = []
        seen_filings = set()

        for score, idx in hits:

            chunk = ordered_chunks[int(idx)]

            filing_key = chunk["doc_url"]

            if unique_filings and filing_key in seen_filings:
                continue

            seen_filings.add(filing_key)

            results.append({
                "similarity_score": round(float(score), 4),
                "chunk_id": chunk["chunk_id"],
                "file_name": chunk["file_name"],
                "deal_name": chunk["deal_name"],
                "filing_date": chunk["filing_date"],
                "filing_type": chunk["filing_type"],
                "filing_year": chunk["filing_year"],
                "doc_url": chunk["doc_url"],
                "text": chunk["text"]
            })

            if len(results) >= top_k:
                break

        return pd.DataFrame(results)

    # Search within a specific filing/year
    if file_name is not None or year is not None:

        eligible_ids = np.array([
            i for i in range(index.ntotal)
            if (
                file_name is None
                or ordered_chunks[i]["file_name"] == file_name
            )
            and (
                year is None
                or int(ordered_chunks[i]["filing_year"]) == int(year)
            )
        ], dtype=np.int64)

        if len(eligible_ids) == 0:
            return pd.DataFrame()

        # Retrieve only eligible vectors
        vectors = index.reconstruct_batch(eligible_ids)

        # Cosine similarity (vectors are normalized)
        scores = vectors @ query_vector[0]

        ranking = np.argsort(scores)[::-1]

        hits = [
            (scores[pos], eligible_ids[pos])
            for pos in ranking
        ]

        return format_results(hits)

    # Search across all currently indexed chunks
    search_k = min(
        index.ntotal,
        max(top_k * 20, 100)
    )

    while True
        scores, indices = index.search(
            query_vector,
            search_k
        )

        hits = [
            (score, idx)
            for score, idx in zip(scores[0], indices[0])
            if idx >= 0
        ]

        results = format_results(hits)

        if len(results) >= top_k:
            return results

        if search_k == index.ntotal:
            return results

        search_k = min(index.ntotal, search_k * 2)


SyntaxError: expected ':' (1243133979.py, line 101)

In [149]:
# Verify that the entire Toyota filing is indexed
assert index.ntotal >= len(toyota_chunks), (
    "Finish embedding the Toyota filing first."
)

question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

results = semantic_search(
    question,
    top_k=8,
    file_name=TOYOTA_FILE
)

print("Question:", question)
print("Matching filing:", TOYOTA_FILE)
print("Retrieved passages:", len(results))

for i, row in results.reset_index(drop=True).iterrows():
    print("RESULT:", i + 1)

    print("Similarity:",
          row["similarity_score"])

    print("Deal:",
          row["deal_name"])

    print("Chunk:",
          row["chunk_id"])

    print("Source:",
          row["doc_url"])

    print("\nRelevant passage:")
    print(row["text"][:1500])


Question: Has legal title to the receivables been transferred via a true sale at each step?
Matching filing: TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08_424H.htm
Retrieved passages: 8
RESULT: 1
Similarity: 0.5792
Deal: TOYOTA AUTO FINANCE RECEIVABLES LLC
Chunk: 0000929638-26-001430_988e346bba_chunk_231
Source: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm

Relevant passage:
the Warranty Receivable under the Receivables Purchase Agreement. For additional information, you should refer to “ Repurchases of Receivables ” and “ Risk Factors—Risks Primarily Related to the Bankruptcy and Insolvency of Transaction Parties and Perfection of Security Interests—The issuing entity’s interests in financed vehicles may be unenforceable or defeated ” in this prospectus. Continuity of Perfection in Financed Vehicles. Under the laws of most states, the perfected security interest in a vehicle would continue for up to four months after the vehicle is moved to a

In [150]:
test_questions = [
    "Has legal title to the receivables been transferred via a true sale at each step?",
    "Who holds the collateral and in what capacity?",
    "How are documents custodied?",
    "Are the files custodied in paper form?",
    "Is the collateral free and clear of liens?",
    "Where are collections initially deposited?",
    "How long can collections stay in a servicer account?"
]

test_results = []

for question in tqdm(
    test_questions,
    desc="Testing semantic retrieval"
):

    results = semantic_search(
        question,
        top_k=5,
        file_name=TOYOTA_FILE
    )

    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1
    ):

        test_results.append({
            "question": question,
            "rank": rank,
            "similarity_score": row["similarity_score"],
            "chunk_id": row["chunk_id"],
            "deal_name": row["deal_name"],
            "source": row["doc_url"],
            "passage": row["text"]
        })

test_results_df = pd.DataFrame(test_results)

display(test_results_df[
    ["question", "rank", "similarity_score", "chunk_id"]
])


Testing semantic retrieval:   0%|          | 0/7 [00:00<?, ?it/s]

,question,rank,similarity_score,chunk_id
0,Has legal title to the receivables been transf...,1,0.5792,0000929638-26-001430_988e346bba_chunk_231
1,Has legal title to the receivables been transf...,2,0.5569,0000929638-26-001430_988e346bba_chunk_230
2,Has legal title to the receivables been transf...,3,0.5538,0000929638-26-001430_988e346bba_chunk_228
3,Has legal title to the receivables been transf...,4,0.5503,0000929638-26-001430_988e346bba_chunk_64
4,Has legal title to the receivables been transf...,5,0.5466,0000929638-26-001430_988e346bba_chunk_124
5,Who holds the collateral and in what capacity?,1,0.5194,0000929638-26-001430_988e346bba_chunk_204
6,Who holds the collateral and in what capacity?,2,0.5096,0000929638-26-001430_988e346bba_chunk_24
7,Who holds the collateral and in what capacity?,3,0.5029,0000929638-26-001430_988e346bba_chunk_207
8,Who holds the collateral and in what capacity?,4,0.4870,0000929638-26-001430_988e346bba_chunk_31
9,Who holds the collateral and in what capacity?,5,0.4869,0000929638-26-001430_988e346bba_chunk_205


In [151]:
retrieval_path = (
    processed_data / "toyota_semantic_retrieval_results.csv"
)

test_results_df.to_csv(
    retrieval_path,
    index=False
)

print("Saved retrieval results to:", retrieval_path)
print("Questions tested:",
      test_results_df["question"].nunique())
print("Total retrieved passages:",
      len(test_results_df))


Saved retrieval results to: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/toyota_semantic_retrieval_results.csv
Questions tested: 7
Total retrieved passages: 35


In [152]:
# Review important Toyota chunks identified during testing
review_chunks = {
    "True sale": [62, 63, 228, 229, 239],
    "Custody": [64, 65, 207],
    "Cash collections": [208, 212]
}

toyota_lookup = {
    c["chunk_index"]: c
    for c in toyota_chunks
}

for category, chunk_numbers in review_chunks.items():
    print("CATEGORY:", category)

    for chunk_num in chunk_numbers:
        chunk = toyota_lookup[chunk_num]

        print(f"\nChunk {chunk_num}")
        print("Source:", chunk["doc_url"])
        print("Text:")
        print(chunk["text"])
        print()


CATEGORY: True sale

Chunk 62
Source: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm
Text:
information systems, including in connection with any consolidation of or change in servicing operations, could have an adverse effect on your notes. TMCC relies on its own information systems and third-party information systems to manage its operations, which creates meaningful operational risk for TMCC. Any failure or interruption of TMCC’s information systems or the third-party information systems on which it relies as a result of inadequate or failed processes or systems, human errors, employee misconduct, cyber-security attacks, catastrophic events, extreme weather conditions, security breaches, acts of vandalism, computer viruses, malware, ransomware, misplaced or lost data, or other events could disrupt TMCC’s normal operating procedures, damage its reputation and have an adverse effect on TMCC’s business, results of operations and financial conditio

In [153]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build a keyword-based index for Toyota's chunks
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

toyota_texts = [c["text"] for c in toyota_chunks]

tfidf_matrix = vectorizer.fit_transform(toyota_texts)

# Use the question we already tested
question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

# Keyword-based ranking
query_tfidf = vectorizer.transform([question])

keyword_scores = (
    tfidf_matrix @ query_tfidf.T
).toarray().ravel()

keyword_order = np.argsort(keyword_scores)[::-1][:20]

# Load saved semantic results without another API call
saved_results = pd.read_csv(
    processed_data / "toyota_semantic_retrieval_results.csv"
)

semantic_results = saved_results[
    saved_results["question"] == question
].sort_values("rank")

# Reciprocal Rank Fusion combines the two rankings
RRF_K = 60
combined_scores = {}

for rank, (_, row) in enumerate(
    semantic_results.iterrows(), start=1
):
    chunk_id = row["chunk_id"]
    combined_scores[chunk_id] = (
        combined_scores.get(chunk_id, 0)
        + 1 / (RRF_K + rank)
    )

for rank, idx in enumerate(keyword_order, start=1):
    chunk_id = toyota_chunks[idx]["chunk_id"]

    combined_scores[chunk_id] = (
        combined_scores.get(chunk_id, 0)
        + 1 / (RRF_K + rank)
    )

# Rank combined results
hybrid_ranking = sorted(
    combined_scores,
    key=combined_scores.get,
    reverse=True
)[:10]

# Inspect final ranking
chunk_lookup = {
    c["chunk_id"]: c for c in toyota_chunks
}

for rank, chunk_id in enumerate(hybrid_ranking, start=1):
    chunk = chunk_lookup[chunk_id]

    print(f"\nResult {rank}")
    print("Chunk:", chunk_id)
    print("Hybrid score:", round(combined_scores[chunk_id], 4))
    print("Preview:", chunk["text"][:500])



Result 1
Chunk: 0000929638-26-001430_988e346bba_chunk_231
Hybrid score: 0.0315
Preview: the Warranty Receivable under the Receivables Purchase Agreement. For additional information, you should refer to “ Repurchases of Receivables ” and “ Risk Factors—Risks Primarily Related to the Bankruptcy and Insolvency of Transaction Parties and Perfection of Security Interests—The issuing entity’s interests in financed vehicles may be unenforceable or defeated ” in this prospectus. Continuity of Perfection in Financed Vehicles. Under the laws of most states, the perfected security interest in

Result 2
Chunk: 0000929638-26-001430_988e346bba_chunk_230
Hybrid score: 0.0308
Preview: and the Indenture Trustee in the vehicle could be defeated through fraud or forgery by the vehicle owner or the Servicer or administrative error by state or local agencies because neither the Issuing Entity nor the Indenture Trustee will be listed as lienholder on the certificates of title. For example, the State of New

In [154]:
# Evaluate hybrid retrieval using saved semantic results
# No API calls required

def hybrid_search_cached(question, top_k=10):

    # Retrieve previously saved semantic results
    semantic_hits = saved_results[
        saved_results["question"] == question
    ].sort_values("rank")

    # Generate TF-IDF scores locally
    query_vector = vectorizer.transform([question])

    keyword_scores = (
        tfidf_matrix @ query_vector.T
    ).toarray().ravel()

    keyword_order = np.argsort(
        keyword_scores
    )[::-1]

    combined_scores = {}
    RRF_K = 60

    # Semantic ranking
    for _, row in semantic_hits.iterrows():

        chunk_id = row["chunk_id"]
        rank = int(row["rank"])

        combined_scores[chunk_id] = (
            combined_scores.get(chunk_id, 0)
            + 1 / (RRF_K + rank)
        )

    # Keyword ranking, ignoring zero-score matches
    keyword_rank = 0

    for idx in keyword_order:

        if keyword_scores[idx] <= 0:
            break

        keyword_rank += 1

        chunk_id = toyota_chunks[idx]["chunk_id"]

        combined_scores[chunk_id] = (
            combined_scores.get(chunk_id, 0)
            + 1 / (RRF_K + keyword_rank)
        )

        if keyword_rank >= 20:
            break

    # Sort final rankings
    ranked_ids = sorted(
        combined_scores,
        key=combined_scores.get,
        reverse=True
    )[:top_k]

    chunk_lookup = {
        c["chunk_id"]: c
        for c in toyota_chunks
    }

    results = []

    for rank, chunk_id in enumerate(ranked_ids, 1):

        chunk = chunk_lookup[chunk_id]

        results.append({
            "question": question,
            "rank": rank,
            "chunk_id": chunk_id,
            "hybrid_score": combined_scores[chunk_id],
            "passage": chunk["text"],
            "doc_url": chunk["doc_url"]
        })

    return pd.DataFrame(results)


# Run all seven previously tested questions
all_hybrid_results = []

for question in saved_results["question"].dropna().unique():

    results = hybrid_search_cached(
        question,
        top_k=10
    )

    all_hybrid_results.append(results)

hybrid_eval_df = pd.concat(
    all_hybrid_results,
    ignore_index=True
)

display(
    hybrid_eval_df[
        ["question", "rank", "chunk_id", "hybrid_score"]
    ]
)

print("Questions tested:",
      hybrid_eval_df["question"].nunique())

print("Total retrieved passages:",
      len(hybrid_eval_df))

,question,rank,chunk_id,hybrid_score
0,Has legal title to the receivables been transf...,1,0000929638-26-001430_988e346bba_chunk_231,0.031545
1,Has legal title to the receivables been transf...,2,0000929638-26-001430_988e346bba_chunk_230,0.030835
2,Has legal title to the receivables been transf...,3,0000929638-26-001430_988e346bba_chunk_228,0.030798
3,Has legal title to the receivables been transf...,4,0000929638-26-001430_988e346bba_chunk_64,0.030118
4,Has legal title to the receivables been transf...,5,0000929638-26-001430_988e346bba_chunk_63,0.016393
...,...,...,...,...
65,How long can collections stay in a servicer ac...,6,0000929638-26-001430_988e346bba_chunk_209,0.015625
66,How long can collections stay in a servicer ac...,7,0000929638-26-001430_988e346bba_chunk_211,0.015625
67,How long can collections stay in a servicer ac...,8,0000929638-26-001430_988e346bba_chunk_220,0.015385
68,How long can collections stay in a servicer ac...,9,0000929638-26-001430_988e346bba_chunk_215,0.015385


Questions tested: 7
Total retrieved passages: 70


In [155]:
hybrid_path = (
    processed_data / "toyota_hybrid_retrieval_results.csv"
)

hybrid_eval_df["relevance_label"] = ""
hybrid_eval_df["review_notes"] = ""

if not hybrid_path.exists():
    hybrid_eval_df.to_csv(hybrid_path, index=False)
    print("Saved:", hybrid_path)
else:
    print("Existing review file preserved:", hybrid_path)


Existing review file preserved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/toyota_hybrid_retrieval_results.csv


In [156]:
# Compare semantic and hybrid rankings
# No API calls

semantic_ranks = saved_results[
    ["question", "chunk_id", "rank"]
].rename(columns={"rank": "semantic_rank"})

hybrid_ranks = hybrid_eval_df[
    ["question", "chunk_id", "rank", "hybrid_score"]
].rename(columns={"rank": "hybrid_rank"})

comparison_df = pd.merge(
    semantic_ranks,
    hybrid_ranks,
    on=["question", "chunk_id"],
    how="outer"
)

# Attach the original prospectus passages
chunk_lookup = {
    c["chunk_id"]: c
    for c in toyota_chunks
}

comparison_df["passage"] = comparison_df[
    "chunk_id"
].map(lambda x: chunk_lookup[x]["text"])

comparison_df["relevance_label"] = ""
comparison_df["review_notes"] = ""

comparison_df = comparison_df.sort_values(
    ["question", "hybrid_rank"],
    na_position="last"
)

review_path = (
    processed_data / "toyota_retrieval_comparison.csv"
)

if not review_path.exists():
    comparison_df.to_csv(review_path, index=False)
    print("Saved:", review_path)
else:
    print("Existing review file preserved.")

print("Questions:", comparison_df["question"].nunique())
print("Passages to review:", len(comparison_df))

display(comparison_df[
    ["question", "chunk_id",
     "semantic_rank", "hybrid_rank"]
].head(20))


Existing review file preserved.
Questions: 7
Passages to review: 70


,question,chunk_id,semantic_rank,hybrid_rank
3,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_207,2.0,1
1,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_187,4.0,2
6,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_266,3.0,3
9,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_65,1.0,4
7,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_279,NaN,5
4,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_214,NaN,6
5,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_220,NaN,7
2,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_190,5.0,8
8,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_64,NaN,9
0,Are the files custodied in paper form?,0000929638-26-001430_988e346bba_chunk_121,NaN,10


In [157]:
question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

review = comparison_df[
    comparison_df["question"] == question
].sort_values("hybrid_rank")

for _, row in review.head(10).iterrows():

    print("Chunk:", row["chunk_id"])
    print("Semantic rank:", row["semantic_rank"])
    print("Hybrid rank:", row["hybrid_rank"])
    print("\nPassage:")
    print(row["passage"])


Chunk: 0000929638-26-001430_988e346bba_chunk_231
Semantic rank: 1.0
Hybrid rank: 1

Passage:
the Warranty Receivable under the Receivables Purchase Agreement. For additional information, you should refer to “ Repurchases of Receivables ” and “ Risk Factors—Risks Primarily Related to the Bankruptcy and Insolvency of Transaction Parties and Perfection of Security Interests—The issuing entity’s interests in financed vehicles may be unenforceable or defeated ” in this prospectus. Continuity of Perfection in Financed Vehicles. Under the laws of most states, the perfected security interest in a vehicle would continue for up to four months after the vehicle is moved to and re-registered by its owner in a state that is different from the one in which it is initially registered. A majority of states generally require surrender of a certificate of title to re-register a vehicle. In those states that require a secured party to hold possession of the certificate of title to maintain perfection of 

In [158]:
# Uses existing results; no API calls

question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

# Provisional relevance labels from passage review
# 2 = directly relevant
# 1 = partially relevant
# 0 = not directly relevant

labels = {
    231: 0,
    230: 0,
    228: 2,
    64: 1,
    63: 2,
    239: 2,
    73: 1,
    229: 2,
    124: 1,
    62: 2
}

review = comparison_df[
    comparison_df["question"] == question
].copy()

review["chunk_number"] = (
    review["chunk_id"]
    .str.extract(r"_chunk_(\d+)$")
    .astype(int)
)

review["relevance"] = review[
    "chunk_number"
].map(labels)

assert review["relevance"].notna().all(), (
    "Some retrieved chunks still need relevance labels."
)

for method, rank_column in [
    ("Semantic", "semantic_rank"),
    ("Hybrid", "hybrid_rank")
]:
    top5 = (
        review.dropna(subset=[rank_column])
        .sort_values(rank_column)
        .head(5)
    )

    direct_hits = (top5["relevance"] == 2).sum()
    precision = direct_hits / len(top5)

    print(f"{method} Precision@5: {precision:.2f}")
    print(f"Direct evidence hits: {direct_hits}/5\n")


Semantic Precision@5: 0.20
Direct evidence hits: 1/5

Hybrid Precision@5: 0.40
Direct evidence hits: 2/5



In [159]:
from collections import Counter

# Count chunks per filing
total_per_file = Counter(
    c["file_name"] for c in ordered_chunks
)

indexed_per_file = Counter(
    c["file_name"]
    for c in ordered_chunks[:index.ntotal]
)

coverage = []

for file_name, total in total_per_file.items():
    indexed = indexed_per_file[file_name]

    coverage.append({
        "file_name": file_name,
        "indexed_chunks": indexed,
        "total_chunks": total,
        "coverage_pct": round(indexed / total * 100, 2),
        "fully_indexed": indexed == total
    })

coverage_df = pd.DataFrame(coverage)

print("Total filings:", len(coverage_df))
print("Fully indexed filings:",
      coverage_df["fully_indexed"].sum())

print("Partially indexed filings:",
      ((coverage_df["indexed_chunks"] > 0) &
       (~coverage_df["fully_indexed"])).sum())

print("Filings not yet indexed:",
      (coverage_df["indexed_chunks"] == 0).sum())

display(
    coverage_df[
        coverage_df["indexed_chunks"] > 0
    ].sort_values(
        "coverage_pct", ascending=False
    )
)


Total filings: 122
Fully indexed filings: 12
Partially indexed filings: 1
Filings not yet indexed: 109


,file_name,indexed_chunks,total_chunks,coverage_pct,fully_indexed
0,TOYOTA_AUTO_FINANCE_RECEIVABLES_LLC_2026-04-08...,312,312,100.00,True
1,AFS_SENSUB_CORP__2024-05-15_424H.htm,206,206,100.00,True
2,AFS_SENSUB_CORP__2024-06-26_424H.htm,214,214,100.00,True
3,AFS_SENSUB_CORP__2024-10-02_424H.htm,214,214,100.00,True
4,AFS_SENSUB_CORP__2025-01-06_424H.htm,211,211,100.00,True
5,AFS_SENSUB_CORP__2025-05-01_424H.htm,226,226,100.00,True
6,AFS_SENSUB_CORP__2025-07-03_424H.htm,217,217,100.00,True
7,AFS_SENSUB_CORP__2025-10-23_424H.htm,199,199,100.00,True
8,AFS_SENSUB_CORP__2026-01-05_424H.htm,211,211,100.00,True
9,AFS_SENSUB_CORP__2026-04-03_424H.htm,196,196,100.00,True


In [160]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Only the chunks with saved embeddings
indexed_chunks = ordered_chunks[:index.ntotal]

# Exclude partially indexed documents by default
complete_files = set(
    coverage_df.loc[
        coverage_df["fully_indexed"], "file_name"
    ]
)

# Create TF-IDF index for keyword retrieval
global_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=150000
)

global_tfidf = global_vectorizer.fit_transform(
    [c["text"] for c in indexed_chunks]
)

# Reuse saved OpenAI vectors
indexed_vectors = index.reconstruct_n(
    0, index.ntotal
)

print("Indexed chunks:", len(indexed_chunks))
print("Fully indexed filings:", len(complete_files))
print("TF-IDF shape:", global_tfidf.shape)
print("Embedding matrix:", indexed_vectors.shape)

Indexed chunks: 2808
Fully indexed filings: 12
TF-IDF shape: (2808, 125136)
Embedding matrix: (2808, 1536)


In [161]:
def hybrid_search(
    question,
    top_k=8,
    file_name=None,
    year=None,
    unique_filings=False,
    include_partial=False
):
    # Select eligible chunks
    eligible = np.array([
        i for i, c in enumerate(indexed_chunks)
        if (
            include_partial
            or c["file_name"] in complete_files
        )
        and (
            file_name is None
            or c["file_name"] == file_name
        )
        and (
            year is None
            or int(c["filing_year"]) == int(year)
        )
    ], dtype=np.int64)

    if len(eligible) == 0:
        return pd.DataFrame()

    # Generate question embedding (one API request)
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[question]
    )

    query_embedding = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_embedding)

    # Semantic similarity
    semantic_scores = (
        indexed_vectors[eligible] @ query_embedding[0]
    )

    semantic_order = np.argsort(
        semantic_scores
    )[::-1][:50]

    # Keyword similarity
    query_tfidf = global_vectorizer.transform(
        [question]
    )

    keyword_scores = (
        global_tfidf[eligible] @ query_tfidf.T
    ).toarray().ravel()

    keyword_order = np.argsort(
        keyword_scores
    )[::-1]

    # Reciprocal Rank Fusion
    rrf_scores = {}
    RRF_K = 60

    for rank, pos in enumerate(semantic_order, 1):
        idx = int(eligible[pos])
        rrf_scores[idx] = (
            rrf_scores.get(idx, 0)
            + 1 / (RRF_K + rank)
        )

    keyword_rank = 0

    for pos in keyword_order:
        if keyword_scores[pos] <= 0:
            break

        keyword_rank += 1
        idx = int(eligible[pos])

        rrf_scores[idx] = (
            rrf_scores.get(idx, 0)
            + 1 / (RRF_K + keyword_rank)
        )

        if keyword_rank >= 50:
            break

    # Final ranking
    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )

    results = []
    seen_files = set()

    for idx in ranked_indices:
        chunk = indexed_chunks[idx]

        if unique_filings:
            if chunk["doc_url"] in seen_files:
                continue

        seen_files.add(chunk["doc_url"])

        results.append({
            "chunk_id": chunk["chunk_id"],
            "file_name": chunk["file_name"],
            "deal_name": chunk["deal_name"],
            "filing_date": chunk["filing_date"],
            "doc_url": chunk["doc_url"],
            "hybrid_score": round(
                rrf_scores[idx], 6
            ),
            "text": chunk["text"]
        })

        if len(results) >= top_k:
            break

    return pd.DataFrame(results)

In [162]:
question = (
    "Has legal title to the receivables been "
    "transferred via a true sale at each step?"
)

toyota_results = hybrid_search(
    question,
    file_name=TOYOTA_FILE,
    top_k=10
)

display(
    toyota_results[
        ["chunk_id", "hybrid_score"]
    ]
)

,chunk_id,hybrid_score
0,0000929638-26-001430_988e346bba_chunk_231,0.031319
1,0000929638-26-001430_988e346bba_chunk_228,0.031025
2,0000929638-26-001430_988e346bba_chunk_230,0.030622
3,0000929638-26-001430_988e346bba_chunk_229,0.030536
4,0000929638-26-001430_988e346bba_chunk_64,0.029911
5,0000929638-26-001430_988e346bba_chunk_63,0.028950
6,0000929638-26-001430_988e346bba_chunk_235,0.028595
7,0000929638-26-001430_988e346bba_chunk_239,0.027505
8,0000929638-26-001430_988e346bba_chunk_74,0.027364
9,0000929638-26-001430_988e346bba_chunk_125,0.026857


In [163]:
cross_deal_question = (
    "Which deals describe third-party custody "
    "or electronic control of receivables?"
)

cross_deal_results = hybrid_search(
    cross_deal_question,
    top_k=8,
    unique_filings=True
)

display(
    cross_deal_results[
        [
            "deal_name",
            "filing_date",
            "hybrid_score",
            "chunk_id"
        ]
    ]
)

# Inspect supporting evidence
for _, row in cross_deal_results.iterrows():
    print("Deal:", row["deal_name"])
    print("Date:", row["filing_date"])
    print("Source:", row["doc_url"])
    print("Passage:", row["text"][:1000])

,deal_name,filing_date,hybrid_score,chunk_id
0,TOYOTA AUTO FINANCE RECEIVABLES LLC,2026-04-08,0.032522,0000929638-26-001430_988e346bba_chunk_65
1,AMERICAN EXPRESS CREDIT ACCOUNT MASTER TRUST,2024-04-11,0.026199,0001193125-24-092834_606c3ec708_chunk_40
2,AFS SENSUB CORP,2025-01-06,0.015385,0001193125-25-002125_5bcf3f4681_chunk_145
3,AFS SENSUB CORP,2024-05-15,0.014925,0001193125-24-139394_6460085a42_chunk_146
4,AFS SENSUB CORP,2026-04-03,0.014706,0001193125-26-142046_331464f52f_chunk_129
5,AFS SENSUB CORP,2024-10-02,0.014493,0001193125-24-231284_dc3a6dab17_chunk_144
6,AFS SENSUB CORP,2024-06-26,0.014085,0001193125-24-169068_b2ca7363dd_chunk_145
7,AFS SENSUB CORP,2026-01-05,0.013889,0001193125-26-002340_ed1eab1425_chunk_145


Deal: TOYOTA AUTO FINANCE RECEIVABLES LLC
Date: 2026-04-08
Source: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm
Passage: contract if (a) there is a “single authoritative copy” of the electronic contract that is readily distinguishable from all other copies and which identifies TMCC as the owner, (b) all other copies of the electronic contract indicate that they are not the “authoritative copy” of the electronic contract, (c) any revisions to the authoritative copy of the electronic contract are readily identifiable as either authorized or unauthorized revisions, and (d) authorized revisions of the electronic contract cannot be made without TMCC’s participation. However, it is possible that another person could acquire an interest in an electronic contract that is superior to TMCC’s interest (and accordingly, the issuing entity’s interest). This could occur if TMCC ceases to have “control” over the electronic contract that is maintained by TMCC 

In [164]:
pilot_results = pd.concat(
    [
        toyota_results.assign(
            question=question,
            search_type="deal_specific"
        ),
        cross_deal_results.assign(
            question=cross_deal_question,
            search_type="cross_deal"
        )
    ],
    ignore_index=True
)

output_path = (
    processed_data /
    "milestone2_task3_hybrid_pilot.csv"
)

pilot_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Total results:", len(pilot_results))


Saved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/milestone2_task3_hybrid_pilot.csv
Total results: 18


In [165]:
review_df = cross_deal_results.copy().reset_index(drop=True)

review_df.insert(
    0, "rank",
    range(1, len(review_df) + 1)
)

# Labels to fill in after inspecting the full passages:
# 2 = directly relevant
# 1 = partially relevant
# 0 = irrelevant

review_df["relevance_label"] = pd.NA
review_df["review_notes"] = ""

review_path = (
    processed_data /
    "cross_deal_retrieval_review.csv"
)

if not review_path.exists():
    review_df.to_csv(review_path, index=False)
    print("Review template saved:", review_path)
else:
    print("Existing review file preserved:", review_path)

# Print full passages for manual review
for _, row in review_df.iterrows():
    print("RANK:", row["rank"])
    print("DEAL:", row["deal_name"])
    print("DATE:", row["filing_date"])
    print("CHUNK:", row["chunk_id"])
    print("SOURCE:", row["doc_url"])
    print("\nFULL PASSAGE:")
    print(row["text"])


Existing review file preserved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/cross_deal_retrieval_review.csv
RANK: 1
DEAL: TOYOTA AUTO FINANCE RECEIVABLES LLC
DATE: 2026-04-08
CHUNK: 0000929638-26-001430_988e346bba_chunk_65
SOURCE: https://www.sec.gov/Archives/edgar/data/1131131/000092963826001430/taot2026-b_424h.htm

FULL PASSAGE:
contract if (a) there is a “single authoritative copy” of the electronic contract that is readily distinguishable from all other copies and which identifies TMCC as the owner, (b) all other copies of the electronic contract indicate that they are not the “authoritative copy” of the electronic contract, (c) any revisions to the authoritative copy of the electronic contract are readily identifiable as either authorized or unauthorized revisions, and (d) authorized revisions of the electronic contract cannot be made without TMCC’s participation. However, it is possible that another person coul

In [166]:
cross_review = cross_deal_results.copy().reset_index(drop=True)

cross_review.insert(
    0, "rank",
    range(1, len(cross_review) + 1)
)

# Preliminary labels based on manual passage review
# 2 = Directly relevant
# 1 = Partially relevant
# 0 = Irrelevant

labels = {
    1: 2,
    2: 0,
    3: 2,
    4: 2,
    5: 2,
    6: 2,
    7: 2,
    8: 2
}

assert len(cross_review) == 8, (
    "Expected 8 results. Verify rankings before applying labels."
)

cross_review["relevance_label"] = (
    cross_review["rank"].map(labels)
)

# Calculate precision at k
def precision_at_k(df, k):
    top_k = df.sort_values("rank").head(k)

    relevant = (
        top_k["relevance_label"] == 2
    ).sum()

    return relevant / len(top_k)

print("Cross-deal retrieval evaluation")
print("-" * 40)

print("Precision@5:",
      round(precision_at_k(cross_review, 5), 3))

print("Precision@8:",
      round(precision_at_k(cross_review, 8), 3))

# Save labels without overwriting the original review
evaluation_path = (
    processed_data /
    "cross_deal_retrieval_labeled.csv"
)

cross_review.to_csv(
    evaluation_path,
    index=False
)

print("\nSaved:", evaluation_path)

display(
    cross_review[
        ["rank", "deal_name", "filing_date",
         "relevance_label", "chunk_id"]
    ]
)

Cross-deal retrieval evaluation
----------------------------------------
Precision@5: 0.8
Precision@8: 0.875

Saved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/cross_deal_retrieval_labeled.csv


,rank,deal_name,filing_date,relevance_label,chunk_id
0,1,TOYOTA AUTO FINANCE RECEIVABLES LLC,2026-04-08,2,0000929638-26-001430_988e346bba_chunk_65
1,2,AMERICAN EXPRESS CREDIT ACCOUNT MASTER TRUST,2024-04-11,0,0001193125-24-092834_606c3ec708_chunk_40
2,3,AFS SENSUB CORP,2025-01-06,2,0001193125-25-002125_5bcf3f4681_chunk_145
3,4,AFS SENSUB CORP,2024-05-15,2,0001193125-24-139394_6460085a42_chunk_146
4,5,AFS SENSUB CORP,2026-04-03,2,0001193125-26-142046_331464f52f_chunk_129
5,6,AFS SENSUB CORP,2024-10-02,2,0001193125-24-231284_dc3a6dab17_chunk_144
6,7,AFS SENSUB CORP,2024-06-26,2,0001193125-24-169068_b2ca7363dd_chunk_145
7,8,AFS SENSUB CORP,2026-01-05,2,0001193125-26-002340_ed1eab1425_chunk_145


In [ ]:
# only run this cells if Task 1 was reran and the FAISS index was rebuilt
import json
import hashlib
import faiss

saved_state = json.loads(state_path.read_text())
saved_index = faiss.read_index(str(index_path))

current_source_hash = hashlib.sha256(
    chunk_path.read_bytes()
).hexdigest()

current_order_hash = hashlib.sha256(
    "\n".join(c["chunk_id"] for c in ordered_chunks)
    .encode("utf-8")
).hexdigest()

print("Source hash matches:",
      saved_state["source_hash"] == current_source_hash)

print("Chunk order matches:",
      saved_state["order_hash"] == current_order_hash)

print("Saved embeddings:", saved_index.ntotal)
print("Checkpoint embeddings:", saved_state["indexed_chunks"])


Source hash matches: True
Chunk order matches: True
Saved embeddings: 2808
Checkpoint embeddings: 2808


In [ ]:
# Only run this cell if underlying chunk text hasn't changed. 
import json
import shutil

# Verify checkpoint consistency
assert saved_state["order_hash"] == current_order_hash
assert saved_index.ntotal == saved_state["indexed_chunks"]
assert saved_index.d == EMBEDDING_DIM

# Back up original checkpoint
backup_path = state_path.with_name(
    "abs_openai_semantic_state_backup.json"
)

if not backup_path.exists():
    shutil.copy2(state_path, backup_path)

# Update source hash
saved_state["source_hash"] = current_source_hash

state_path.write_text(
    json.dumps(saved_state, indent=2)
)

print("Checkpoint updated successfully!")
print("Embeddings preserved:", saved_index.ntotal)
print("Backup saved:", backup_path)


Checkpoint updated successfully!
Embeddings preserved: 2808
Backup saved: /Users/mahisheth/Downloads/BTT-PIMCO1C/PIMCO-1C-ai-surveillance-analysis-based-on-abs-prospectuses/data/processed/abs_openai_semantic_state_backup.json
